# PolyChain v29: Polymer Property Prediction
**ANRF AISEHack 2.0 - Polymer Property Prediction**

8-model ensemble + stacking + multi-task. Features: target transforms (Yeo-Johnson/RankGauss), topological graph invariants, GNN embeddings, multi-seed MLP, Huber loss.

Run all cells in order. Skips completed work on re-run.

In [ ]:
# Cell 1: Install dependencies
import sys, os, subprocess, shutil, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
print(f"Python: {sys.version[:6]}")

def detect_gpu():
    try:
        out = subprocess.check_output(
            "nvidia-smi --query-gpu=name --format=csv,noheader",
            shell=True, timeout=10
        ).decode().strip()
        return out[:60]
    except Exception:
        return "CPU"

gpu = detect_gpu()
print(f"GPU: {gpu}")

# Install PyTorch (version depends on GPU)
try:
    import torch
    print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
except ImportError:
    print("Installing PyTorch...")
    if "P100" in gpu:
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "torch==2.5.1+cu121", "--index-url",
                        "https://download.pytorch.org/whl/cu121", "-q"],
                       capture_output=True)
    elif "T4" in gpu:
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "torch==2.6.0+cu124", "--index-url",
                        "https://download.pytorch.org/whl/cu124", "-q"],
                       capture_output=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch", "-q"],
                       capture_output=True)
    import torch
    print(f"PyTorch {torch.__version__}")

# Install PyG
pyg_check = subprocess.run([sys.executable, "-c", "import torch_geometric"],
                           capture_output=True)
if pyg_check.returncode != 0:
    print("Installing PyTorch Geometric...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"],
                   capture_output=True)

# Install other deps
for pkg in ["rdkit", "xgboost", "lightgbm", "catboost", "optuna",
            "scikit-learn", "pandas", "numpy", "pyyaml", "pyarrow"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True)

print("All dependencies ready!")

In [ ]:
# Cell 2: Clone repo (main branch) and setup
WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/NotShubham1112/Poly.git"
BRANCH = "main"

os.chdir(WORK_DIR)
REPO_DIR = os.path.join(WORK_DIR, "Poly")

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.check_call(["git", "checkout", BRANCH],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(["git", "pull", "origin", BRANCH],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.chdir(WORK_DIR)
else:
    subprocess.check_call([
        "git", "clone", "-b", BRANCH, "--single-branch", REPO_URL
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Copy polymer_competition contents to work dir
SRC = os.path.join(REPO_DIR, "polymer_competition")
for item in os.listdir(SRC):
    src_path = os.path.join(SRC, item)
    dst_path = os.path.join(WORK_DIR, item)
    if os.path.exists(dst_path):
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dst_path)
    else:
        shutil.copy2(src_path, dst_path)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {len(os.listdir('.'))} items")

In [ ]:
# Cell 3: Build features + CV splits (initial pass, no GNN embeddings yet)
!python -m data.generate_splits --config config.yaml --target tg
!python -m data.generate_splits --config config.yaml --target egc
!python -m features.build_features --config config.yaml
print("Features and CV splits ready!")

In [ ]:
# Cell 4: Train tree models 5-fold, both targets
# These run fast on CPU, leaving GPU free for GNNs
!python -m training.run_all_folds --model xgb --config config.yaml
!python -m training.run_all_folds --model lgb --config config.yaml
!python -m training.run_all_folds --model catboost --config config.yaml
!python -m training.run_all_folds --model rf --config config.yaml
print("Tree models complete!")

In [ ]:
# Cell 5: Train GNNs + auto-extract embeddings, 5-fold both targets
# --target flag triggers embedding extraction during training
!python -m training.train --model_type gcn --target tg --fold all
!python -m training.train --model_type gcn --target egc --fold all
!python -m training.train --model_type gat --target tg --fold all
!python -m training.train --model_type gat --target egc --fold all
!python -m training.train --model_type mpnn --target tg --fold all
!python -m training.train --model_type mpnn --target egc --fold all
print("GNN training + embedding extraction complete!")

In [ ]:
# Cell 6: Rebuild features with GNN embeddings enabled
# Embeddings saved to features/embeddings/{exp_ver}_{target}/
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["features"]["use_embeddings"] = True
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

!python -m features.build_features --config config.yaml
print("Features rebuilt with GNN embeddings!")

In [ ]:
# Cell 7: Train MLP 5-seed ensemble with GNN embeddings + Huber loss
!python -m training.train --model_type mlp --target tg --fold all --n_seeds 5 --loss huber
!python -m training.train --model_type mlp --target egc --fold all --n_seeds 5 --loss huber
print("MLP multi-seed training complete!")

In [ ]:
# Cell 8: Multi-task training (80 epochs, dual-output head)
# Jointly predicts Tg and Egc with uncertainty-weighted loss
!python run_multitask.py --epochs 80
print("Multi-task training complete!")

In [ ]:
# Cell 9: Build weighted ensemble + Level-2 stacking per target
!python -m ensemble.build_ensemble --config config.yaml --target tg
!python -m ensemble.build_ensemble --config config.yaml --target egc
print("Weighted ensembles built!")

!python -m ensemble.stacking_ensemble --config config.yaml --target tg
!python -m ensemble.stacking_ensemble --config config.yaml --target egc
print("Stacking ensembles built!")

In [ ]:
# Cell 10: Generate submission (pick best per target)
!python run_submission.py
print("Submission generated!")

In [ ]:
# Cell 11: Show model performance summary
import json
from pathlib import Path

manifest_path = Path("experiments/manifest.json")
if manifest_path.exists():
    data = json.loads(manifest_path.read_text())

    from collections import defaultdict
    results = defaultdict(lambda: defaultdict(list))
    for entry in data:
        if entry.get("status") == "completed":
            model = entry.get("model_type", "?")
            target = entry.get("target", "?")
            score = entry.get("score", 0)
            results[model][target].append(score)

    print(f"{'Model':<20} {'TG R2':>10} {'EGC R2':>10} {'Mean R2':>10}")
    print("-" * 52)
    for model in sorted(results.keys()):
        tg_scores = results[model].get("tg", [0])
        egc_scores = results[model].get("egc", [0])
        tg_mean = sum(tg_scores) / len(tg_scores) if tg_scores else 0
        egc_mean = sum(egc_scores) / len(egc_scores) if egc_scores else 0
        mean_r2 = (tg_mean + egc_mean) / 2
        print(f"{model:<20} {tg_mean:>10.4f} {egc_mean:>10.4f} {mean_r2:>10.4f}")

    tg_models = [m for m in results if 'tg' in results[m]]
    egc_models = [m for m in results if 'egc' in results[m]]
    tg_best = max(results[m]['tg'] for m in tg_models) if tg_models else 0
    egc_best = max(results[m]['egc'] for m in egc_models) if egc_models else 0
    print(f"\nBest single model TG: {tg_best:.4f}")
    print(f"Best single model EGC: {egc_best:.4f}")
    print(f"Mean of best: {(tg_best + egc_best) / 2:.4f}")
else:
    print("No experiment manifest found.")

In [ ]:
# Cell 12: Copy submission to Kaggle output
import shutil
shutil.copy("outputs/submissions/submission.csv", "/kaggle/working/submission.csv")
print("Copied submission.csv to /kaggle/working/")
print("Submit this file to the competition!")